# 🧠 คำอธิบายและตัวอย่างการปฏิบัติการเพื่อนบ้านใกล้สุด K คน (K-Nearest Neighbors - KNN)

ยินดีต้อนรับสู่โน้ตบุ๊กประกอบการอธิบายเรื่อง **K-Nearest Neighbors (KNN)**! ในโน้ตบุ๊กนี้เราจะ:
1. สร้างชุดข้อมูลจำลอง 2 มิติที่แทนเวกเตอร์ฝังตัว (Visual Feature Embeddings) ของกล่องวัตถุระหว่าง `lever-handle` (มือหมุนแบบก้านโยก) และ `handwheel-handle` (มือหมุนแบบล้อกลม)
2. จำแนกประเภทข้อมูลโดยใช้คลาสมาตรฐาน `KNeighborsClassifier` ของ `scikit-learn`
3. พล็อตกราฟแสดงขอบเขตการตัดสินใจ (Decision Boundaries) ของค่า $K$ ต่างๆ ($K=1$, $K=5$, $K=20$) เพื่อทำความเข้าใจประเด็น Bias-Variance Tradeoff
4. เขียนระบบ **KNN จากศูนย์ (from scratch)** ด้วยภาษา Python/NumPy โดยอ้างอิงตามระยะทางยูคลิเดียน (Euclidean Distance) และการลงคะแนนเสียงข้างมาก (Majority Voting)
5. ประเมินผลโค้ดที่เราเขียนขึ้นเองเทียบกับไลบรารีมาตรฐาน scikit-learn
6. อธิบายการใช้งาน KNN ในระบบ Computer Vision สมัยใหม่ (เช่น ระบบจำตัวบุคคลซ้ำ (Re-identification) และฐานข้อมูลเวกเตอร์)

เริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันก่อนครับ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from collections import Counter

# กำหนดค่า seed เพื่อให้ได้ผลลัพธ์การสุ่มเหมือนเดิมทุกครั้ง
np.random.seed(42)

## 1. การสร้างข้อมูลตามกรณีศึกษา (Case Study Data Generation)

เราจะจำลองคุณลักษณะของเวกเตอร์ฝังตัว (2D Bounding Box Visual Embeddings) จำนวน 80 ตัวอย่าง:
*   Class 1 (`lever-handle` มือหมุนแบบก้านโยก): จุดศูนย์กลางของการกระจายตัวอยู่ที่ (2.0, 3.0)
*   Class 0 (`handwheel-handle` มือหมุนแบบล้อกลม): จุดศูนย์กลางของการกระจายตัวอยู่ที่ (4.5, 4.0)

In [ ]:
m = 80

# คลาส 1: มือหมุนแบบก้านโยก (Lever Handles)
X_lever = np.random.randn(m // 2, 2) * 0.8 + np.array([2.0, 3.0])
y_lever = np.ones(m // 2)

# คลาส 0: มือหมุนแบบล้อกลม (Handwheel Handles)
X_handwheel = np.random.randn(m // 2, 2) * 0.8 + np.array([4.5, 4.0])
y_handwheel = np.zeros(m // 2)

# รวมชุดข้อมูล
X_train = np.vstack((X_lever, X_handwheel))
y_train = np.concatenate((y_lever, y_handwheel))

# พล็อตกราฟจุดกระจายตัวของข้อมูล
plt.figure(figsize=(8, 5))
plt.scatter(X_lever[:, 0], X_lever[:, 1], color='blue', label='Class 1: Lever Handle', alpha=0.7)
plt.scatter(X_handwheel[:, 0], X_handwheel[:, 1], color='red', label='Class 0: Handwheel Handle', alpha=0.7)
plt.xlabel('Embedding Feature 1')
plt.ylabel('Embedding Feature 2')
plt.title('Bounding Box Visual Embedding Space')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 2. การวิเคราะห์ขอบเขตการตัดสินใจ (Bias-Variance Tradeoff)

การเลือกค่าเพื่อนบ้านใกล้สุด $K$ ส่งผลต่อโมเดลอย่างมาก เราลองมาเทรนแบบจำลองด้วย scikit-learn ที่ค่า $K=1$, $K=5$ และ $K=20$ เพื่อดูผลลัพธ์ของเส้นขอบเขตการตัดสินใจกันครับ

In [ ]:
# สร้างตาราง Grid เพื่อนำมาพล็อตพื้นที่แบ่งแยกดินแดนขอบเขต
x_min, x_max = X_train[:, 0].min() - 1, X_train[:, 0].max() + 1
y_min, y_max = X_train[:, 1].min() - 1, X_train[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05),
                     np.arange(y_min, y_max, 0.05))
grid_points = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cmap_light = ListedColormap(['#FFAAAA', '#AAAAFF'])
cmap_bold = ['#FF0000', '#0000FF']

K_values = [1, 5, 20]

for idx, K in enumerate(K_values):
    clf = KNeighborsClassifier(n_neighbors=K)
    clf.fit(X_train, y_train)
    
    Z = clf.predict(grid_points)
    Z = Z.reshape(xx.shape)
    
    ax = axes[idx]
    ax.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.6)
    ax.scatter(X_train[:, 0], X_train[:, 1], c=[cmap_bold[int(i)] for i in y_train], edgecolor='k', s=40)
    ax.set_title(f'KNN Boundary (K={K})')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

*   **$K=1$ (โอเวอร์ฟิต - Overfitting):** ขอบเขตการตัดสินใจมีความหยักโค้งสูงมาก พยายามโอบล้อมทุกจุดข้อมูลที่เป็นสิ่งรบกวน ส่งผลให้มีความแปรปรวน (Variance) สูง
*   **$K=5$ (ความพอดีที่เหมาะสม - Ideal Fit):** เส้นขอบเขตมีความราบเรียบและไหลลื่น แบ่งแยกประเภทโดยรวมได้ดีในขณะที่ละเว้นจุดสัญญาณรบกวนขนาดเล็ก
*   **$K=20$ (อันเดอร์ฟิต - Underfitting):** ขอบเขตง่ายเกินไปจนเป็นเส้นแบ่งหยาบๆ ทำให้สูญเสียพฤติกรรมการแบ่งประเภทสำหรับรายละเอียดคุณลักษณะของบางพื้นที่ไป

## 3. การสร้าง KNN จากศูนย์ด้วย NumPy (KNN from Scratch)

เรามาสร้างตัวทำนาย KNN กันครับ โดยตัวโมเดลจะจดจำชุดข้อมูลฝึกสอนไว้ จากนั้นเมื่อมีข้อมูลส่งเข้ามาเพื่อสอบถาม (Query Point):
1.  คำนวณระยะทางยูคลิเดียน (Euclidean Distance) ระหว่างข้อมูลสอบถามไปยังข้อมูลฝึกสอนทั้งหมด
2.  จัดเรียงลำดับระยะทางจากน้อยไปหามาก
3.  เลือกป้ายกำกับคลาส (Labels) ของจุดข้อมูลที่ใกล้ที่สุดจำนวน $K$ ตัวแรก
4.  หาคำตอบขั้นสุดท้ายด้วยการลงคะแนนเสียงข้างมาก (Majority Voting)

In [ ]:
class CustomKNN:
    def __init__(self, K=5):
        self.K = K
        self.X_train = None
        self.y_train = None
        
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
        
    def _predict_single(self, x_query):
        # 1. คำนวณระยะทางยูคลิเดียน (Euclidean distance)
        distances = np.sqrt(np.sum((self.X_train - x_query) ** 2, axis=1))
        
        # 2. หาดัชนีของจุดข้อมูลที่มีระยะทางสั้นที่สุด K ลำดับแรก
        k_indices = np.argsort(distances)[:self.K]
        
        # 3. ดึงป้ายกำกับคลาสของเพื่อนบ้านที่ใกล้ที่สุด K ตัวแรก
        k_nearest_labels = self.y_train[k_indices].astype(int)
        
        # 4. ลงคะแนนเสียงเลือกคลาสที่เป็นส่วนใหญ่
        most_common = Counter(k_nearest_labels).most_common(1)
        return most_common[0][0]
        
    def predict(self, X_queries):
        return np.array([self._predict_single(x) for x in X_queries])

# ทดสอบแบบจำลอง KNN ที่เราเขียนขึ้นเอง
scratch_knn = CustomKNN(K=5)
scratch_knn.fit(X_train, y_train)

# เทรนแบบจำลองของ scikit-learn เพื่อเทียบผล
sklearn_knn = KNeighborsClassifier(n_neighbors=5)
sklearn_knn.fit(X_train, y_train)

# ทดสอบทำนายจุดข้อมูลเป้าหมายใหม่
test_points = np.array([
    [1.5, 2.5],  # อยู่ลึกในพื้นที่คลาส 1
    [4.0, 4.5],  # อยู่ลึกในพื้นที่คลาส 0
    [3.1, 3.5]   # บริเวณชายขอบก้ำกึ่ง
])

preds_scratch = scratch_knn.predict(test_points)
preds_sklearn = sklearn_knn.predict(test_points)

print("เปรียบเทียบผลการทำนาย:")
for i, pt in enumerate(test_points):
    print(f"Point {pt} | Scratch Pred: {preds_scratch[i]} | Sklearn Pred: {preds_sklearn[i]}")

## 4. การตรวจสอบความถูกต้องของแบบจำลอง (Performance Validation)

เรามาประเมินหาค่าความแม่นยำ (Accuracy) ของ Custom KNN ที่เราสร้างขึ้นบนชุดข้อมูลทั้งหมดเปรียบเทียบกับไลบรารีมาตรฐาน

In [ ]:
y_pred_scratch = scratch_knn.predict(X_train)
y_pred_sklearn = sklearn_knn.predict(X_train)

accuracy_scratch = accuracy_score(y_train, y_pred_scratch)
accuracy_sklearn = accuracy_score(y_train, y_pred_sklearn)

print(f"Custom Scratch KNN Accuracy: {accuracy_scratch * 100:.2f}%")
print(f"Scikit-Learn KNN Accuracy  : {accuracy_sklearn * 100:.2f}%")

## 💡 ความเชื่อมโยงสู่ Deep Learning และ YOLO
*   **การระบุตัวตนวัตถุซ้ำ (Object Re-Identification - Re-ID):** ในงานติดตามวัตถุหลายชิ้น (Multi-Object Tracking) เช่น การติดตามบุคคลหรือรถยนต์ผ่านกล้องหลายตัว โครงข่ายประสาทระดับลึกจะทำหน้าที่สกัดเวกเตอร์ฝังตัว (Feature Embeddings) จากกล่องวัตถุ Bounding Box จากนั้น อัลกอริทึมติดตามจะคำนวณระยะห่างแบบโคไซน์ (Cosine Distance) หรือระยะทางยูคลิเดียน ระหว่างพารามิเตอร์ของกล่องที่ตรวจเจอใหม่กับกล่องในอดีตเพื่อยืนยันว่าเป็นวัตถุตัวเดิม ซึ่งแนวคิดนี้คือการค้นหาเพื่อนบ้านใกล้สุดตัวแรก หรือ 1-Nearest Neighbor Search นั่นเอง!
*   **การค้นหาและฐานข้อมูลเวกเตอร์ (Vector Search & Databases):** ระบบค้นคืนปัญญาประดิษฐ์ยุคใหม่ (เช่น RAG หรือการค้นหาภาพถ่ายด้วยภาพ) มีการใช้งานฐานข้อมูลเวกเตอร์แบบพิเศษ (เช่น Milvus, Pinecone, หรือ FAISS) เพื่อรันคำค้นหาประเภท KNN ความเร็วสูงผ่านเวกเตอร์ที่มีมิติสูงหลายล้านชุดพร้อมกัน